# RetailMart Lakehouse

## Notebook : 03_Load_Order_Items

### Layer
Bronze Layer

### Objective

This notebook ingests the Order Items dataset from the Raw Volume into the Bronze layer.

### Pipeline Steps

- Read Raw Order Items Dataset
- Perform Data Profiling
- Validate Schema
- Perform Data Quality Checks
- Perform Business Validations
- Add Audit Columns
- Load into Bronze Delta Table
- Verify Data Load
- Generate Bronze Load Report

### Source
Raw Volume

### Target

retailmart.bronze.order_items

In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)

In [0]:
RUN_ID = generate_run_id()
START_TIME = start_pipeline()
PIPELINE_NAME = "Bronze_Order_Items"
SOURCE_FILE = RAW_ITEMS
TARGET_TABLE = TARGET_TABLE_ITEM

Pipeline Started : 2026-07-15 07:37:11.688943


In [0]:
# Explicit Schema 
order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

In [0]:
order_items_df = (
    spark.read
         .schema(order_items_schema)
         .option("header", True)
         .csv(SOURCE_FILE)
)

In [0]:
total_rows = order_items_df.count()

total_columns = len(order_items_df.columns)

print("ORDER ITEMS DATASET PROFILE")

print(f"Rows    : {total_rows}")
print(f"Columns : {total_columns}")

order_items_df.printSchema()

display(order_items_df.limit(10))

ORDER ITEMS DATASET PROFILE
Rows    : 86328
Columns : 7
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
ORD_0000001,1,PROD_001950,SELL_0224,2023-06-18T14:30:00.000Z,1754.7,20.17
ORD_0000002,1,PROD_000989,SELL_0405,2021-06-14T11:02:00.000Z,2105.2,45.48
ORD_0000003,1,PROD_000254,SELL_0283,2022-04-03T19:38:00.000Z,1627.94,78.29
ORD_0000004,1,PROD_001327,SELL_0437,2021-09-12T22:29:00.000Z,65.22,31.08
ORD_0000004,2,PROD_002714,SELL_0396,2021-09-12T22:29:00.000Z,946.39,44.43
ORD_0000004,3,PROD_001507,SELL_0446,2021-09-12T22:29:00.000Z,157.96,55.93
ORD_0000005,1,PROD_001872,SELL_0152,2022-08-28T07:46:00.000Z,1681.83,14.0
ORD_0000006,1,PROD_001518,SELL_0490,2023-05-29T16:27:00.000Z,856.74,64.83
ORD_0000007,1,PROD_001504,SELL_0324,2021-09-22T15:50:00.000Z,373.8,66.88
ORD_0000008,1,PROD_001133,SELL_0241,2021-01-21T14:24:00.000Z,899.98,23.95


In [0]:
expected_columns = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value"
]

# Schema Validation

In [0]:
schema_status = validate_schema(order_items_df,order_items_schema)

Schema Validation Passed


In [0]:
# PRIMARY KEY IS - (order_id, order_item_id)

In [0]:
pk_status = validate_primary_key(
    order_items_df,
    ["order_id", "order_item_id"]
)

Total Rows : 86328
Distinct Count : 86328
Primary Key Validation Passed — (order_id, order_item_id)


In [0]:
duplicate_rows = duplicate_summary(
    order_items_df,
    total_rows
)

Duplicate Rows : 0


In [0]:
null_summary(order_items_df)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,0,0,0,0,0,0


In [0]:
# Invalid price where price<=0
invalid_price = order_items_df.filter(
    F.col("price") <= 0
).count()

print(f"Invalid Price Records : {invalid_price}")

Invalid Price Records : 0


In [0]:
# Freight Validation
invalid_freight = order_items_df.filter(
    F.col("freight_value") < 0
).count()

print(f"Negative Freight Records : {invalid_freight}")

Negative Freight Records : 0


In [0]:
order_items_df.select(
    F.min("price").alias("Minimum Price"),
    F.max("price").alias("Maximum Price"),
    F.avg("price").alias("Average Price"),
     F.sum("price").alias("Total Revenue")
).show()

+-------------+-------------+------------------+--------------------+
|Minimum Price|Maximum Price|     Average Price|       Total Revenue|
+-------------+-------------+------------------+--------------------+
|        15.02|      2499.96|1258.1847297516483|1.0861657135000029E8|
+-------------+-------------+------------------+--------------------+



In [0]:
orders_without_items = (
    order_items_df
        .groupBy("order_id")
        .count()
        .filter(F.col("count") == 0)
        .count()
)

print(f"Orders Without Items : {orders_without_items}")

Orders Without Items : 0


In [0]:
order_items_df = add_audit_columns(
    order_items_df,
    PIPELINE_NAME,
    RUN_ID
)

In [0]:
status, error = write_bronze_table(
    order_items_df,
    TARGET_TABLE
)

Bronze table written: retailmart.bronze.order_items


# Verification

bronze_df = spark.table(TARGET_TABLE)

rows_written = bronze_df.count()

print("VERIFICATION")

print(f"Rows Written : {rows_written}")

if rows_written == total_rows:
    print("Verification Successful")
else:
    print("Row Count Mismatch")

In [0]:
bronze_df = spark.table(TARGET_TABLE)

rows_written = bronze_df.count()

display(bronze_df.limit(5))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp,ingestion_date,pipeline_name,run_id
ORD_0000001,1,PROD_001950,SELL_0224,2023-06-18T14:30:00.000Z,1754.7,20.17,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000002,1,PROD_000989,SELL_0405,2021-06-14T11:02:00.000Z,2105.2,45.48,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000003,1,PROD_000254,SELL_0283,2022-04-03T19:38:00.000Z,1627.94,78.29,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000004,1,PROD_001327,SELL_0437,2021-09-12T22:29:00.000Z,65.22,31.08,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000004,2,PROD_002714,SELL_0396,2021-09-12T22:29:00.000Z,946.39,44.43,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e


In [0]:
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_FILE,
    target=TARGET_TABLE,
    rows_read=total_rows,
    rows_written=rows_written,
    duplicate_count=duplicate_rows,
    start_time=START_TIME,
    status=status
)

BRONZE LOAD REPORT
Pipeline        : Bronze_Order_Items
Run ID          : 89cd67fb-0173-4983-9c48-0c154f057a6e
Source          : /Volumes/dbacademy/default/raw/raw_items_dataset.csv
Target          : retailmart.bronze.order_items
Rows Read       : 86328
Rows Written    : 86328
Duplicate Rows  : 0
Start Time      : 2026-07-15 07:37:11.688943
End Time        : 2026-07-15 07:38:23.355670
Duration (sec)  : 71.67
Status          : SUCCESS


In [0]:
# Validation Summary

validation_summary = spark.createDataFrame(

    [

        ("Schema Validation",
         "PASS"),

        ("Composite PK Validation",
         "PASS" if pk_status else "FAIL"),

        ("Duplicate Validation",
         "PASS" if duplicate_rows == 0 else "FAIL"),

        ("Price Validation",
         "PASS" if invalid_price == 0 else "FAIL"),

        ("Freight Validation",
         "PASS" if invalid_freight == 0 else "FAIL")

    ],

    ["Validation", "Status"]

)

display(validation_summary)

Validation,Status
Schema Validation,PASS
Composite PK Validation,PASS
Duplicate Validation,PASS
Price Validation,PASS
Freight Validation,PASS


# Engineering Observations

### Summary

- Schema successfully validated against the expected structure.
- Composite Primary Key `(order_id, order_item_id)` verified.
- No duplicate records detected.
- Price and Freight Value validations completed.
- Bronze Delta table created successfully.